# GraphRAG Agent — Text2Cypher 자가교정 (LangGraph)

**GraphRAG** 는 지식을 **그래프(노드/관계)** 로 표현한 DB(예: Neo4j)에서 정보를 검색해 답하는 RAG 다. 문서 청크 대신 "엔티티와 관계"를 다뤄, "A 와 함께 출연한 배우" 같은 **관계형 질문** 에 강하다.

이 노트북은 앞서 배운 **SQL RAG 의 그래프DB 버전** 이다. 자연어를 **Cypher**(Neo4j 쿼리 언어)로 바꿔 실행하고, 틀리면 스스로 고친다. LangGraph 로 다음 흐름을 구현한다:

```
START → guardrails ─(영화질문?)─▶ generate_cypher → validate ─(에러?)─▶ correct_cypher
           │(무관)                                      │(정상)          │
           ▼                                    execute_cypher ◀──────────┘
   generate_final_answer ◀─(관련)─ relevance ◀────┘   └(무관)─▶ correct_cypher
           │
          END
```

> **Neo4j 필요.** 이 노트북은 영화 그래프(Person-ACTED_IN->Movie-IN_GENRE->Genre) 를 가진 Neo4j 인스턴스에 연결한다. README 의 'Neo4j 준비' 를 먼저 보라. `OPENAI_API_KEY` 도 필요.

## Neo4j 연결

`.env` 에 접속 정보를 넣는다. Neo4j Aura(클라우드 무료) 또는 로컬/샌드박스 인스턴스 사용.
```
NEO4J_URI=neo4j+s://xxxx.databases.neo4j.io   # 또는 neo4j://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your-password
OPENAI_API_KEY=sk-...
```

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"

URI = os.environ["NEO4J_URI"]
AUTH = (os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"])

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()   # 연결 확인
print("Neo4j 연결 성공")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")

## DB 스키마 추출

Cypher 를 생성하려면 LLM 이 **DB 구조(노드 라벨/속성/관계)** 를 알아야 한다. Neo4j 내장 프로시저로 스키마를 문자열로 뽑아 프롬프트에 넣는다.

In [ ]:
from collections import defaultdict

def get_schema() -> str:
    schema = ""
    with driver.session() as session:
        # 노드 라벨 + 속성
        node_schema = session.run(
            "CALL db.schema.nodeTypeProperties() YIELD nodeType, propertyName, propertyTypes "
            "RETURN nodeType, propertyName, propertyTypes"
        )
        nodes = defaultdict(dict)
        for r in node_schema:
            label = r["nodeType"].replace(":", "")
            nodes[label][r["propertyName"]] = r["propertyTypes"][0] if r["propertyTypes"] else "UNKNOWN"

        # 관계 방향
        rel_types = session.run(
            "MATCH (a)-[r]->(b) RETURN DISTINCT labels(a) AS f, type(r) AS t, labels(b) AS to"
        )
        rels = set()
        for r in rel_types:
            rels.add(f"(:{r['f'][0]})-[:{r['t']}]->(:{r['to'][0]})")

    schema += "\nNode properties:\n"
    for label, props in nodes.items():
        schema += f"{label} {{{', '.join(f'{k}: {v}' for k, v in props.items())}}}\n"
    schema += "\nThe relationships:\n"
    for rel in sorted(rels):
        schema += rel + "\n"
    return schema

print(get_schema())

## Graph State
[basics] 입력/전체/출력 State 를 분리한다. `steps` 는 리듀서로 실행 경로를 누적한다.

In [ ]:
from operator import add
from typing import Annotated, List
from typing_extensions import TypedDict

class InputState(TypedDict):
    question: str

class OverallState(TypedDict):
    question: str
    next_action: str
    cypher_statement: str
    cypher_errors: List[str]
    database_records: List[dict]
    steps: Annotated[List[str], add]

class OutputState(TypedDict):
    answer: str
    steps: List[str]
    cypher_statement: str

## 1) guardrails — 영화 질문인지 판단
[projects] 구조화 출력으로 'movie' / 'end' 분기. 무관한 질문은 바로 종료로 보낸다.

In [ ]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

guardrails_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Decide whether a question is related to movies. If it refers to any movie, actor, "
     "director, film industry or related topics, output 'movie'. Otherwise output 'end'."),
    ("human", "{question}"),
])

class GuardrailsOutput(BaseModel):
    decision: Literal["movie", "end"] = Field(description="movie-related or not")

guardrails_chain = guardrails_prompt | llm.with_structured_output(GuardrailsOutput)

def guardrails(state: InputState) -> OverallState:
    print("-- GUARDRAILS --")
    decision = guardrails_chain.invoke({"question": state["question"]}).decision
    records = None
    if decision == "end":
        records = "This question is not about movies. I cannot answer it."
    return {"next_action": decision, "database_records": records, "steps": ["guardrail"]}

## 2) generate_cypher — 자연어 → Cypher

few-shot 예시 + 스키마를 주고 Cypher 를 생성한다. (SQL RAG 의 `generate_query` 와 같은 발상)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

examples = [
    {"question": "How many movies has Tom Hanks acted in?",
     "query": "MATCH (a:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m)"},
    {"question": "List all the genres of the movie Schindler's List",
     "query": "MATCH (m:Movie {title: 'Schindler\\'s List'})-[:IN_GENRE]->(g:Genre) RETURN g.name"},
    {"question": "Find the actor with the highest number of movies.",
     "query": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie) RETURN a.name, COUNT(m) AS c ORDER BY c DESC LIMIT 1"},
]

text2cypher_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Convert the question to a Cypher query. No preamble. "
     "Do not wrap in backticks. Respond with a Cypher statement only!"),
    ("human",
     "You are a Neo4j expert. Create a syntactically correct Cypher query.\n"
     "Schema:\n{schema}\n\nExamples:\n{fewshot_examples}\n\n"
     "User input: {question}\nCypher query:"),
])
text2cypher_chain = text2cypher_prompt | llm | StrOutputParser()

def generate_cypher(state: OverallState) -> OverallState:
    print("-- GENERATE CYPHER --")
    cypher = text2cypher_chain.invoke({
        "question": state["question"],
        "fewshot_examples": str(examples),
        "schema": get_schema(),
    })
    print("Generated:", cypher)
    return {"cypher_statement": cypher, "steps": ["generate_cypher"]}

## 3) validate_cypher — 문법·스키마 검사

두 단계 검사: ① Neo4j `EXPLAIN` 으로 문법 오류 잡기, ② LLM 으로 스키마 정합성(없는 라벨/속성/관계) 점검. 오류가 있으면 `correct_cypher`, 없으면 `execute_cypher` 로.

In [ ]:
from typing import Optional
from neo4j.exceptions import CypherSyntaxError

validate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Cypher expert reviewing a statement written by a junior developer."),
    ("human",
     "Check for syntax errors, missing/undefined variables, labels/relationships/properties "
     "not in the schema, and whether it can answer the question.\n"
     "Schema:\n{schema}\n\nQuestion: {question}\n\nCypher: {cypher}"),
])

class ValidateCypherOutput(BaseModel):
    errors: Optional[List[str]] = Field(
        default=None, description="syntax/semantic errors; explain schema-cypher discrepancies")

validate_chain = validate_prompt | llm.with_structured_output(ValidateCypherOutput)

def validate_cypher(state: OverallState) -> OverallState:
    print("-- VALIDATE CYPHER --")
    cypher = state["cypher_statement"]
    errors = []
    # 1) 문법 검사 (EXPLAIN 은 실행 없이 파싱만)
    try:
        driver.execute_query(f"EXPLAIN {cypher}")
    except CypherSyntaxError as e:
        errors.append(e.message)
    # 2) 스키마 정합성 (LLM)
    llm_out = validate_chain.invoke(
        {"question": state["question"], "schema": get_schema(), "cypher": cypher}
    )
    if llm_out.errors:
        errors.extend(llm_out.errors)
    next_action = "correct_cypher" if errors else "execute_cypher"
    return {"next_action": next_action, "cypher_statement": cypher, "cypher_errors": errors,
            "steps": ["validate_cypher"]}

## 4) correct_cypher — 오류 반영해 재작성
[rag] 에러 메시지 + 스키마를 주고 고친 Cypher 를 받는다 (SQL RAG 의 재생성 루프).

In [ ]:
correct_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Cypher expert. Correct the statement based on the errors. No preamble. "
     "Do not wrap in backticks. Respond with a Cypher statement only!"),
    ("human",
     "Schema:\n{schema}\n\nQuestion: {question}\n\nCypher: {cypher}\n\n"
     "Errors: {errors}\n\nCorrected Cypher statement:"),
])
correct_chain = correct_prompt | llm | StrOutputParser()

def correct_cypher(state: OverallState) -> OverallState:
    print("-- CORRECT CYPHER --")
    corrected = correct_chain.invoke({
        "question": state["question"], "errors": state["cypher_errors"],
        "cypher": state["cypher_statement"], "schema": get_schema(),
    })
    print("Corrected:", corrected)
    return {"cypher_statement": corrected, "steps": ["correct_cypher"]}

## 5) execute_cypher — 쿼리 실행

In [ ]:
no_results = "I couldn't find any relevant information in the database"

def execute_cypher(state: OverallState) -> OverallState:
    print("-- EXECUTE CYPHER --")
    try:
        with driver.session(database="neo4j") as session:
            records = session.execute_read(
                lambda tx: tx.run(state["cypher_statement"]).data())
    except Exception as e:
        records = str(e)
    return {"database_records": records if records else no_results,
            "next_action": "end", "steps": ["execute_cypher"]}

## 6) relevance — 결과가 질문에 충분한지 평가
충분하면 답변 생성, 부족하면 Cypher 재작성으로 (환각 방지).

In [ ]:
relevance_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Judge whether the database results contain enough information to fully answer the question. "
     "Binary decision."),
    ("human",
     "Return score 'yes'(sufficient) or 'no', and brief feedback.\n\n"
     "Question: {question}\n\nResults: {results}"),
])

class RelevanceScore(BaseModel):
    score: str = Field(description="'yes' or 'no'")
    feedback: str = Field(description="why sufficient or not")

relevance_chain = relevance_prompt | llm.with_structured_output(RelevanceScore)

def relevance(state: OverallState) -> OverallState:
    print("-- GRADE RELEVANCE --")
    result = relevance_chain.invoke(
        {"question": state["question"], "results": state["database_records"]})
    next_action = "generate_final_answer" if result.score == "yes" else "correct_cypher"
    return {"next_action": next_action, "database_records": state["database_records"],
            "cypher_errors": result.feedback, "steps": ["relevance"]}

## 7) generate_final_answer — 결과로 답변 생성

In [ ]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human",
     "Use the database results to give a succinct, definitive answer.\n\n"
     "Results: {results}\nQuestion: {question}"),
])
final_chain = final_prompt | llm | StrOutputParser()

def generate_final_answer(state: OverallState) -> OutputState:
    print("-- GENERATE FINAL ANSWER --")
    answer = final_chain.invoke(
        {"question": state["question"], "results": state["database_records"]})
    return {"answer": answer, "steps": ["generate_final_answer"]}

## 8) 조건부 엣지

In [ ]:
def guardrails_condition(state: OverallState) -> Literal["generate_cypher", "generate_final_answer"]:
    return "generate_final_answer" if state["next_action"] == "end" else "generate_cypher"

def validate_condition(state: OverallState) -> Literal["correct_cypher", "execute_cypher"]:
    return state["next_action"]   # 'correct_cypher' or 'execute_cypher'

def relevance_condition(state: OverallState) -> Literal["generate_final_answer", "correct_cypher"]:
    return state["next_action"]

## 그래프 조립
```
guardrails →(movie) generate_cypher → validate →(정상) execute → relevance →(관련) final → END
         →(end) final                        →(에러) correct → validate ...      →(무관) correct
```

In [ ]:
from langgraph.graph import END, START, StateGraph

gb = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
gb.add_node(guardrails)
gb.add_node(generate_cypher)
gb.add_node(validate_cypher)
gb.add_node(correct_cypher)
gb.add_node(execute_cypher)
gb.add_node(relevance)
gb.add_node(generate_final_answer)

gb.add_edge(START, "guardrails")
gb.add_conditional_edges("guardrails", guardrails_condition)
gb.add_edge("generate_cypher", "validate_cypher")
gb.add_conditional_edges("validate_cypher", validate_condition)
gb.add_edge("correct_cypher", "validate_cypher")   # 수정 후 재검증
gb.add_edge("execute_cypher", "relevance")
gb.add_conditional_edges("relevance", relevance_condition)
gb.add_edge("generate_final_answer", END)
graph = gb.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 테스트
### 무관한 질문 → guardrails 가 바로 종료

In [ ]:
graph.invoke({"question": "What's the weather in Spain?"})

### 영화 관련 질문 → Cypher 생성·실행·답변

In [ ]:
graph.invoke({"question": "Casino 영화에 출연한 배우들은 누구인가요?"})

## 정리

- **GraphRAG** = 지식 그래프(Neo4j)에서 관계형 정보를 검색해 답하는 RAG
- **Text2Cypher**: 자연어 → Cypher 쿼리 (SQL RAG 의 그래프DB 버전)
- 자가교정 루프: 생성 → **문법·스키마 검증** → 실행 → **관련성 평가** → (부족하면 재작성)
- guardrails 로 범위 밖 질문을 미리 거름
- 앞서 배운 RAG 품질 관리(검증/자가교정)가 그래프DB 에도 그대로 적용된다

이로써 강의의 마지막 주제 GraphRAG 까지 다뤘다. 문서(RAG) → DB(SQL RAG) → 그래프(GraphRAG) 로 '외부 지식을 근거로 답하는' 에이전트의 스펙트럼을 완성했다.